In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q sentencepiece datasets
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/otto_gpt'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/ckpt', exist_ok=True)
print("프로젝트 폴더:", PROJECT_DIR)
print("내용:", os.listdir(PROJECT_DIR))

In [ ]:
from datasets import load_dataset

# ── 사전학습 코퍼스 규모 (스케일업) ──────────────────────────────────
# 57M 모델의 Chinchilla 적정 학습량 ≈ 11억 토큰. 위키 문서 1개 ≈ 300~500 토큰이라
# 다양성과 분량을 위해 문서 수를 크게 늘린다 (기존 50k → 400k, 약 8~10배).
# 한국어 위키 전체는 약 60만+ 문서.
#
# ⚠ corpus_wiki.txt 가 이미 있으면 재다운로드를 건너뛴다. 규모를 바꾸려면 Drive 의
#    data/corpus_wiki.txt 와 data/train.bin · data/val.bin 을 지우고 다시 실행할 것.
N_WIKI_DOCS = 400_000

RAW_TXT = f'{PROJECT_DIR}/data/corpus_wiki.txt'

if not os.path.exists(RAW_TXT):
    print(f"한국어 위키 다운로드 중... (목표 {N_WIKI_DOCS:,} 문서)")
    ds = load_dataset("wikimedia/wikipedia", "20231101.ko",
                      split="train", streaming=True)
    kept = 0
    with open(RAW_TXT, 'w', encoding='utf-8') as f:
        for doc in ds:
            if kept >= N_WIKI_DOCS:
                break
            text = doc['text'].strip()
            if len(text) > 100:          # 너무 짧은 문서 제외
                f.write(text + '\n\n')
                kept += 1
                if kept % 50_000 == 0:
                    print(f"  {kept:,} 문서 저장")
    print(f"완료: {RAW_TXT} ({kept:,} 문서)")
else:
    print("이미 존재(재다운로드 건너뜀):", RAW_TXT)

size_mb = os.path.getsize(RAW_TXT) / 1e6
print(f"위키 코퍼스 크기: {size_mb:.1f} MB")

## ➕ CulturaX 추가 = 이어서 학습(continual pretraining)

otto 는 이미 위키로 40k step 학습됨. 새 데이터(maywell/korean_textbooks 등)를 더해 **이어서** 베이스를 넓힌다.
아래 셀에서 새 데이터를 받은 뒤, **이어서 학습하려면 아래 4가지를 지킬 것**:

1. **토크나이저 재학습 금지** — 기존 `tokenizer_ko.model` 그대로 사용(아래 토크나이저 셀이 자동 skip).
   바꾸면 기존 체크포인트가 무효가 된다.
2. **bin 재생성** — Drive 의 `data/train.bin`·`data/val.bin` 을 **삭제** → 토크나이징 셀이
   위키+CulturaX 합본으로 다시 빌드한다(안 지우면 옛 데이터 그대로).
3. **`total_iters` 상향** — 하이퍼파라미터 셀에서 `40_000 → 예) 80_000`. 안 그러면 이미 도달해
   추가 학습이 0 이 된다. 체크포인트(`otto_gpt.pt`)에서 자동 resume.
4. 그대로 학습 셀 실행 → 40k 부터 **넓힌 데이터로 이어서** 학습.

In [ ]:
# ── CulturaX (ko): 대규모 정제 한국어 웹텍스트로 베이스 확장 ──
# ⚠ 게이트 데이터셋. 먼저 (1) https://huggingface.co/datasets/uonlp/CulturaX 에서 'Agree' 클릭,
#    (2) HF 토큰 발급(https://huggingface.co/settings/tokens, read 권한).
USE_CULTURAX = False   # 게이트라 끔. 비게이트 대안은 아래 셀(ADD_EXTRA)
N_CULTURAX_DOCS = 300_000      # 스트리밍 문서 수 (위키 400k 와 합쳐 베이스 다양화)

if USE_CULTURAX:
    from huggingface_hub import login
    # 권장: Colab 좌측 🔑 Secrets 에 HF_TOKEN 저장 후 자동 로그인
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        print("HF 로그인 필요 → 아래 한 줄 주석 해제 후 토큰 입력:")
        # login(token="hf_xxxxx")

    CX_TXT = f"{PROJECT_DIR}/data/corpus_culturax.txt"
    if not os.path.exists(CX_TXT):
        print(f"CulturaX(ko) 스트리밍... (목표 {N_CULTURAX_DOCS:,} 문서)")
        try:
            cx = load_dataset("uonlp/CulturaX", "ko", split="train",
                              streaming=True, token=True)
            kept = 0
            with open(CX_TXT, "w", encoding="utf-8") as f:
                for doc in cx:
                    if kept >= N_CULTURAX_DOCS:
                        break
                    t = (doc.get("text") or "").strip()
                    if len(t) > 200:        # 짧은 웹 노이즈 제외
                        f.write(t + "\n\n")
                        kept += 1
                        if kept % 50_000 == 0:
                            print(f"  {kept:,} 문서")
            print(f"완료: {CX_TXT} ({kept:,} 문서, {os.path.getsize(CX_TXT)/1e6:.1f} MB)")
        except Exception as e:
            print("CulturaX 로드 실패 — 약관 동의/토큰 확인:", str(e)[:200])
    else:
        print("이미 존재:", CX_TXT)
else:
    print("CulturaX 건너뜀 (USE_CULTURAX=False)")

In [ ]:
# ── 비게이트 추가 코퍼스: 위키 외 다양한 한국어 텍스트 (가입/토큰 불필요) ──
# maywell/korean_textbooks(오픈) + KoAlpaca(오픈) → 위키만의 좁음을 넓힌다.
ADD_EXTRA = True
N_EXTRA_DOCS = 500_000   # 소스당 문서 수 (350M 먹이려면 크게 — 다운로드 시간 늘어남)

if ADD_EXTRA:
    # 1) maywell/korean_textbooks — 한국어 교과서/지식 텍스트(게이트 없음)
    for subset in ['claude_evol', 'ko_wikidata']:   # 한국어 위주 서브셋
        OUT = f'{PROJECT_DIR}/data/corpus_{subset}.txt'
        if os.path.exists(OUT):
            print('이미 존재:', os.path.basename(OUT)); continue
        print(f'maywell/korean_textbooks [{subset}] 스트리밍... (목표 {N_EXTRA_DOCS:,})')
        try:
            ds = load_dataset('maywell/korean_textbooks', name=subset, split='train', streaming=True)
            kept = 0
            with open(OUT, 'w', encoding='utf-8') as f:
                for doc in ds:
                    if kept >= N_EXTRA_DOCS: break
                    t = (doc.get('text') or '').strip()
                    if len(t) > 200:
                        f.write(t + '\n\n'); kept += 1
                        if kept % 50_000 == 0: print(f'  {kept:,}')
            print(f'완료: {os.path.basename(OUT)} ({kept:,} 문서, {os.path.getsize(OUT)/1e6:.1f} MB)')
        except Exception as e:
            print(f'[{subset}] 실패(서브셋명 확인):', str(e)[:150])

    # 2) KoAlpaca Q&A 텍스트(대화체, 게이트 없음)
    KA = f'{PROJECT_DIR}/data/corpus_koalpaca.txt'
    if not os.path.exists(KA):
        ka = load_dataset('beomi/KoAlpaca-v1.1a', split='train')
        with open(KA, 'w', encoding='utf-8') as f:
            for ex in ka:
                q=(ex.get('instruction') or '').strip(); a=str(ex.get('output') or '').strip()
                if q and a: f.write(q+'\n'+a+'\n\n')
        print('완료: corpus_koalpaca.txt')
    else:
        print('이미 존재: corpus_koalpaca.txt')
else:
    print('추가 코퍼스 건너뜀 (ADD_EXTRA=False)')

In [ ]:
import glob

all_txt_files = glob.glob(f'{PROJECT_DIR}/data/*.txt')
print("발견된 데이터 파일:")
for p in all_txt_files:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.1f} MB")

MERGED_TXT = f'{PROJECT_DIR}/data/_merged.txt'
with open(MERGED_TXT, 'w', encoding='utf-8') as out:
    for p in all_txt_files:
        if os.path.basename(p) == '_merged.txt':
            continue
        with open(p, encoding='utf-8') as f:
            out.write(f.read() + '\n')
print(f"\n합본: {MERGED_TXT} ({os.path.getsize(MERGED_TXT)/1e6:.1f} MB)")

In [ ]:
import sentencepiece as spm

SPM_PREFIX = f'{PROJECT_DIR}/tokenizer_ko'
VOCAB_SIZE = 16000

if not os.path.exists(SPM_PREFIX + '.model'):
    print("토크나이저 학습 중... (몇 분 소요)")
    spm.SentencePieceTrainer.train(
        input=MERGED_TXT,
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=0.9995,
        
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols=['<sep>', '<pad2>'],
        input_sentence_size=1_000_000,   
        shuffle_input_sentence=True,
    )
    print("완료:", SPM_PREFIX + '.model')
else:
    print("이미 존재:", SPM_PREFIX + '.model')

sp = spm.SentencePieceProcessor(model_file=SPM_PREFIX + '.model')
print("\n어휘 크기:", sp.get_piece_size())
sample = "인공지능은 사람처럼 학습하는 기술이다."
ids = sp.encode(sample)
print(f"\n원문: {sample}")
print(f"토큰: {[sp.id_to_piece(i) for i in ids]}")
print(f"ID:   {ids}")
print(f"복원: {sp.decode(ids)}")

In [ ]:
import numpy as np, glob as _glob

TRAIN_BIN = f'{PROJECT_DIR}/data/train.bin'
VAL_BIN = f'{PROJECT_DIR}/data/val.bin'

# 코퍼스(corpus_*.txt)가 bin 보다 새로우면 재토크나이즈 → CulturaX 추가 등이 자동 반영됨
# (수동으로 bin 삭제할 필요 없음)
_corpus = [p for p in _glob.glob(f'{PROJECT_DIR}/data/*.txt') if not p.endswith('_merged.txt')]
_need = (not (os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN))) or (
    _corpus and max(os.path.getmtime(p) for p in _corpus) > os.path.getmtime(TRAIN_BIN))

if _need:
    print('코퍼스 토크나이징 중... (새 데이터 반영)')
    all_ids = []
    with open(MERGED_TXT, encoding='utf-8') as f:
        buf = []
        for line in f:
            line = line.strip()
            if not line: continue
            ids = sp.encode(line)
            buf.extend(ids); buf.append(sp.eos_id())
            if len(buf) > 1_000_000:
                all_ids.append(np.array(buf, dtype=np.uint16)); buf = []
        if buf: all_ids.append(np.array(buf, dtype=np.uint16))
    data = np.concatenate(all_ids)
    n = len(data); split = int(n*0.9)
    data[:split].tofile(TRAIN_BIN); data[split:].tofile(VAL_BIN)
    print(f'총 {len(data):,} 토큰 → train {split:,} / val {n-split:,} 재생성 완료')
else:
    print(f'최신 bin 존재(코퍼스 변경 없음). train {os.path.getsize(TRAIN_BIN)//2:,} 토큰')

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    block_size: int = 512     
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 624
    dropout: float = 0.1
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.n_head, self.n_embd, self.dropout = cfg.n_head, cfg.n_embd, cfg.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4*cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),   
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),   
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.transformer.wpe.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
            return logits, loss
        logits = self.lm_head(x[:, [-1], :])
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPT 모델 정의 완료")

In [ ]:
# ── 모델 크기: 350M (GPT-2 medium 급) — 파라미터로 천장 올리기 ──
MODEL_TAG = "350m"   # 체크포인트 구분 (기존 57M otto_gpt.pt 는 보존)
config = GPTConfig(
    vocab_size=VOCAB_SIZE,
    block_size=512,
    n_layer=24,
    n_head=16,
    n_embd=1024,
    dropout=0.1,
)

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 38:        # A100 40GB (권장)
        batch_size = 8
        grad_accum = 6
    elif vram_gb >= 22:      # L4 / 3090 24GB
        batch_size = 2
        grad_accum = 24
    else:                    # T4 16GB — 350M엔 빠듯(OOM 가능, 매우 느림)
        batch_size = 1
        grad_accum = 48
else:
    batch_size, grad_accum = 2, 1

learning_rate = 3e-4
min_lr = 3e-5

# ── 학습량 스케일업 ──────────────────────────────────────────────
# total_iters : LR 코사인 감쇠의 '전체 지평'. 여러 세션에 걸쳐 누적 학습해도
#               이 값까지 천천히 감쇠한다. (기존엔 max_iters 를 지평으로 써서
#               5000 step 이후 LR 이 min_lr 로 죽는 버그가 있었음)
# max_iters   : 이번 세션에서 '추가로' 돌릴 step 수 (Colab 끊김 대비 분할 학습).
#               끝나면 노트북을 다시 실행하면 체크포인트에서 이어서 total_iters 까지.
total_iters = 60_000     # 350M 목표 지평 (~15억 토큰). Chinchilla(~70억)엔 부족하나 Colab 현실선
max_iters = 4_000        # 세션당 (350M은 step당 느림 → 자주 끊고 이어서)
warmup_iters = 2_000
eval_interval = 500
eval_iters = 50
weight_decay = 0.1
grad_clip = 1.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'

tokens_per_step = batch_size * grad_accum * config.block_size
print(f"디바이스: {device}, dtype: {dtype}")
print(f"batch_size={batch_size}, grad_accum={grad_accum}")
print(f"유효 배치: {tokens_per_step:,} 토큰/step")
print(f"전체 학습 지평(total_iters)={total_iters:,} → 누적 약 {tokens_per_step*total_iters/1e9:.2f}B 토큰")
print(f"이번 세션 step(max_iters)={max_iters:,}")

# 학습 데이터 토큰 수로 '몇 epoch' 인지 추정 (train.bin 이 있을 때)
if os.path.exists(TRAIN_BIN):
    train_tokens = os.path.getsize(TRAIN_BIN) // 2  # uint16 → 2 bytes
    epochs = tokens_per_step * total_iters / max(train_tokens, 1)
    print(f"\n학습 토큰: {train_tokens:,} → total_iters 기준 약 {epochs:.1f} epoch")
    if epochs > 8:
        print("  ⚠ epoch 과다(>8): 데이터를 더 늘리거나 total_iters 를 줄이세요(과적합 위험).")
    elif epochs < 1.5:
        print("  ⚠ epoch 부족(<1.5): total_iters 를 늘리면 더 학습 가능.")

m_tmp = GPT(config)
print(f"\n모델 파라미터: {m_tmp.num_params()/1e6:.1f}M")
del m_tmp

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode='r')
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - config.block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+config.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+config.block_size].astype(np.int64)) for i in ix])
    if device == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print("배치 shape:", xb.shape, yb.shape)
print("샘플 디코드:", sp.decode(xb[0][:30].tolist()))

In [ ]:
import time

# latest = 이어서 학습용(최근 step), best = 추론용(최저 val_loss).
# 장기 분할 학습에서는 'latest' 에서 이어가야 step 손실이 없다.
CKPT_PATH = f'{PROJECT_DIR}/ckpt/otto_gpt_{MODEL_TAG}.pt'        # latest (resume)
BEST_PATH = f'{PROJECT_DIR}/ckpt/otto_gpt_{MODEL_TAG}_best.pt'   # best val (inference)

model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=weight_decay)
scaler = torch.amp.GradScaler(enabled=(dtype == 'float16'))

iter_num = 0
best_val_loss = float('inf')
if os.path.exists(CKPT_PATH):
    print(f"✓ 체크포인트 발견 → 이어서 학습: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    iter_num = ckpt['iter_num']
    best_val_loss = ckpt['best_val_loss']
    print(f"  복원: iter_num={iter_num:,}, best_val_loss={best_val_loss:.4f}")
else:
    print("새로 학습 시작 (체크포인트 없음)")

print(f"학습 파라미터: {model.num_params()/1e6:.1f}M")
print(f"진행: {iter_num:,} / {total_iters:,} (전체 지평) | 이번 세션 +{max_iters:,} step\n")

def get_lr(it):
    # 코사인 감쇠 지평은 total_iters (세션당 max_iters 가 아님!)
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it >= total_iters:
        return min_lr
    ratio = (it - warmup_iters) / (total_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)) if device=='cuda' else torch.no_grad():
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def save_checkpoint(path, val_loss, tag=""):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'config': config.__dict__,
        'iter_num': iter_num,
        'best_val_loss': best_val_loss,
    }, path)
    print(f"    💾 {tag}체크포인트 저장 (iter={iter_num:,}, val_loss={val_loss:.4f})")

print("학습 시작...\n")
model.train()
t0 = time.time()
target_iter = min(iter_num + max_iters, total_iters)   # 전체 지평을 넘지 않게

while iter_num < target_iter:
    lr = get_lr(iter_num)
    for g in optimizer.param_groups:
        g['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for micro in range(grad_accum):
        X, Y = get_batch('train')
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                _, loss = model(X, Y)
                loss = loss / grad_accum
        else:
            _, loss = model(X, Y)
            loss = loss / grad_accum
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if iter_num % 50 == 0:
        dt = time.time() - t0
        print(f"iter {iter_num:6d} | loss {accum_loss:.4f} | lr {lr:.2e} | {dt:.1f}s")
        t0 = time.time()

    if iter_num % eval_interval == 0 and iter_num > 0:
        losses = estimate_loss()
        print(f"  >> eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
        # latest 는 매 eval 마다 저장(끊김 대비) → 이어서 학습 시 step 손실 없음
        save_checkpoint(CKPT_PATH, losses['val'], tag="latest ")
        # best 는 val 개선 시에만 (추론용)
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            save_checkpoint(BEST_PATH, best_val_loss, tag="best ")

    iter_num += 1

final_losses = estimate_loss()
print(f"\n세션 종료. 최종 val_loss: {final_losses['val']:.4f}")
if final_losses['val'] < best_val_loss:
    best_val_loss = final_losses['val']
    save_checkpoint(BEST_PATH, best_val_loss, tag="best ")
save_checkpoint(CKPT_PATH, final_losses['val'], tag="latest ")
print(f"누적 학습 step: {iter_num:,} / {total_iters:,}")
if iter_num < total_iters:
    print("→ 아직 지평 미도달. 노트북을 다시 실행하면 이어서 학습합니다.")
else:
    print("→ 전체 학습 지평 도달! 추론에는 otto_gpt_best.pt 를 사용하세요.")

In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    model.eval()
    ids = sp.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                out = model.generate(x, max_new_tokens, temperature, top_k)
        else:
            out = model.generate(x, max_new_tokens, temperature, top_k)
    return sp.decode(out[0].tolist())

for prompt in ["인공지능은", "한국의 역사는", "오늘 날씨가"]:
    print(f"[입력] {prompt}")
    print(f"[생성] {generate_text(prompt, max_new_tokens=80)}\n")

In [ ]:
print("="*50)
print("otto-GPT 현재 상태")
print("="*50)
print(f"파라미터:      {model.num_params()/1e6:.1f}M")
print(f"누적 학습 step: {iter_num:,}")
print(f"최고 val_loss:  {best_val_loss:.4f}")
print(f"어휘 크기:      {sp.get_piece_size():,}")
print(f"체크포인트:     {CKPT_PATH}")
print(f"토크나이저:     {SPM_PREFIX}.model")
print("="*50)
print("\n다음에 이어서 학습하려면: 이 노트북을 다시 열고 셀을 순서대로 실행하세요.")